In [2]:
# Import initial libraries
import pandas as pd
import numpy as np
import boto3
from io import StringIO
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# Load your modeling dataset from S3
# Replace the path below with your actual S3 path from Day 26
s3 = boto3.client('s3')
bucket_name = 'cmse492-cranda98-nyc311-964165949460-us-east-1-an' # replace with your bucket name (make sure you have the account regional suffix)
file_name = 'modeling/modeling_data.csv' # make sure to include the path if it's in a folder, e.g. 'modeling/modeling_data.csv'

obj = s3.get_object(Bucket=bucket_name, Key=file_name)
data = obj['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(data))

print(f"Shape: {df.shape}")
df.head()

Shape: (157244, 6)


,borough,incident_zip,agency,day_of_week,hour_of_day,complaint_category
0,QUEENS,11368.0,NYPD,5,21,traffic
1,BRONX,10461.0,NYPD,5,21,traffic
2,BROOKLYN,11208.0,NYPD,5,21,noise
3,BROOKLYN,11228.0,NYPD,5,21,traffic
4,MANHATTAN,10002.0,DOT,5,21,traffic


In [5]:
# Quick data check — confirm the shape, column names, and target variable
print("Columns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)

Columns: ['borough', 'incident_zip', 'agency', 'day_of_week', 'hour_of_day', 'complaint_category']

Missing values:
borough                 0
incident_zip          570
agency                  0
day_of_week             0
hour_of_day             0
complaint_category      0
dtype: int64

Data types:
borough                object
incident_zip          float64
agency                 object
day_of_week             int64
hour_of_day             int64
complaint_category     object
dtype: object


In [10]:
# Define feature columns and target column
feature_cols = ['borough', 'incident_zip', 'agency', 'day_of_week', 'hour_of_day']
target_col   = 'complaint_category'

# Encode categorical columns as numeric
df['borough_enc'] = df['borough'].astype('category').cat.codes
df['agency_enc']  = df['agency'].astype('category').cat.codes

# Update feature_cols to use encoded versions
feature_cols = ['borough', 'incident_zip', 'agency', 'day_of_week', 'hour_of_day']
target_col = 'complaint_category'

X = df[feature_cols]
y = df[target_col]

print(f"Features: {feature_cols}")
print(f"Target: {target_col}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nClass balance: {y.value_counts(normalize=True).round(3).to_dict()}")

Features: ['borough', 'incident_zip', 'agency', 'day_of_week', 'hour_of_day']
Target: complaint_category

Target distribution:
complaint_category
traffic       56813
housing       55794
noise         25195
sanitation    19442
Name: count, dtype: int64

Class balance: {'traffic': 0.361, 'housing': 0.355, 'noise': 0.16, 'sanitation': 0.124}


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:  {X_train.shape[0]} rows")
print(f"Test set:      {X_test.shape[0]} rows")
print(f"\nClass balance in training set:")
print(y_train.value_counts(normalize=True).round(3))
print(f"\nClass balance in test set:")
print(y_test.value_counts(normalize=True).round(3))

Training set:  125795 rows
Test set:      31449 rows

Class balance in training set:
complaint_category
traffic       0.361
housing       0.355
noise         0.160
sanitation    0.124
Name: proportion, dtype: float64

Class balance in test set:
complaint_category
traffic       0.361
housing       0.355
noise         0.160
sanitation    0.124
Name: proportion, dtype: float64


In [19]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

# Pipeline for numeric: impute NaNs first, then scale
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # fills NaN zip codes with median
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
        ('num', num_pipeline, num_cols),
    ]
)

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

ohe_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
encoded_feature_names = ohe_names + num_cols

print(f"\nEncoded feature matrix shape: {X_train_enc.shape}")
print(f"Total features after encoding: {len(encoded_feature_names)}")
print(f"\nFirst few encoded feature names: {encoded_feature_names[:10]}")


Encoded feature matrix shape: (125795, 11)
Total features after encoding: 11

First few encoded feature names: ['borough_BROOKLYN', 'borough_MANHATTAN', 'borough_QUEENS', 'borough_STATEN ISLAND', 'agency_DOT', 'agency_DSNY', 'agency_HPD', 'agency_NYPD', 'incident_zip', 'day_of_week']


In [20]:
# Import the logistic regression model
from sklearn.linear_model import LogisticRegression

# Train the logistic regression baseline (adjust the model type and parameters as needed for your specific task)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_enc, y_train)

print("Model trained successfully.")
print(f"Classes: {model.classes_}")

Model trained successfully.
Classes: ['housing' 'noise' 'sanitation' 'traffic']


In [22]:
# Inspect the model coefficients
# After encoding, use encoded_feature_names instead of feature_cols
coef_df = pd.DataFrame({
    'feature': encoded_feature_names,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

print("Model coefficients (top 10 by magnitude):")
print(coef_df.head(10).to_string(index=False))
print("\nPositive coefficient = feature pushes toward the positive class")
print("Negative coefficient = feature pushes toward the negative class")

Model coefficients (top 10 by magnitude):
              feature  coefficient
           agency_HPD    10.319504
          hour_of_day     0.029310
          day_of_week     0.001669
         incident_zip    -0.060743
    borough_MANHATTAN    -0.268449
     borough_BROOKLYN    -0.433437
       borough_QUEENS    -0.551510
borough_STATEN ISLAND    -0.604209
           agency_DOT    -1.433440
          agency_DSNY    -1.439334

Positive coefficient = feature pushes toward the positive class
Negative coefficient = feature pushes toward the negative class


In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Generate predictions on the test set
y_pred = model.predict(X_test_enc)

# Compute metrics
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")

ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].